In [51]:
## Task 1 – Represent Reviews Using Word2Vec
import pandas as pd
import gensim
import re
from xxlimited import Str

df = pd.read_csv("07-yelp-dataset.txt", sep="\t", names=["review", "label"])

def clean_text(text):
    text = Str(text)
    text = text.lower()
    text = re.sub(r"[0-9]", '', text)
    text = re.sub(r"[)(,”“.’$-]", '', text)
    return text

df['cleaned_text'] = df.apply(clean_text, axis=1)
tokens = df['cleaned_text'].str.split().tolist()
print("Data size:", len(tokens))

from sklearn.model_selection import train_test_split
X = df["review"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

train_tokens = X_train.apply(clean_text).str.split().tolist()
test_tokens = X_test.apply(clean_text).str.split().tolist()

sgm = gensim.models.Word2Vec(train_tokens, min_count=1, vector_size=200, window=5, sg=1)
## 1.1 Training the model on the training data
sgm = gensim.models.Word2Vec(train_tokens, min_count = 1, vector_size = 200, window = 5, sg = 1)

## 1.2 changing str vlues to numpy because pytorch does not support str values
import numpy as np
def numpyfy(words, model):
    vecs = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vecs, axis=0)

## 1.3 converting numpy to tensor (pytorch)
import torch

training_data_numpy = np.array([numpyfy(words, sgm) for words in train_tokens])
print("Training data shape:", training_data_numpy.shape)
testing_data_numpy = np.array([numpyfy(words, sgm) for words in test_tokens])
print("Testing data shape:", testing_data_numpy.shape)

xtrain = torch.tensor(training_data_numpy, dtype=torch.float32)
xtest = torch.tensor(testing_data_numpy, dtype=torch.float32)

ytrain = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)  # Reshape to (n_samples, 1)
ytest_labels = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)  # Reshape to (n_samples, 1)






Data size: 1000
Training reviews: 800
Testing reviews: 200
Training data shape: (800, 200)
Testing data shape: (200, 200)


In [52]:
## Task 2 – Build a Different Neural Network

input_size = xtrain.shape[1]
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(input_size, 128),   # Input layer
    nn.ReLU(),
    
    nn.Linear(128, 64),          
    nn.ReLU(),

    nn.Linear(64, 32),           
    nn.ReLU(),

    nn.Linear(32, 1)             # Output layer
)

print(model)

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)


In [54]:

loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 15

for epoch in range(epochs):
    model.train()

    optimizer.zero_grad()

    # 1. Forward pass
    outputs = model(xtrain)

    # 2. Calculate loss
    loss = loss_function(outputs, ytrain)

    # 3. Backpropagation
    loss.backward()

    # 4. Update model weights
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 5/15 - Loss: 0.6931
Epoch 10/15 - Loss: 0.6930
Epoch 15/15 - Loss: 0.6930


In [56]:
## 9. Evaluate the Model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test = model(xtest)
    test_probabilities = torch.sigmoid(test)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(ytest_labels, predictions)

print(f"Test Accuracy: {accuracy:.3f}")
print(confusion_matrix(ytest_labels, predictions))


Test Accuracy: 0.480
[[ 95   1]
 [103   1]]
